# 01 - Medical images: CT, Hounsfield units, DICOM and NIfTI

In this notebook we open chest CT images and learn what is inside the files.
You only need to run the cells and read. The exercises are at the end.

## Index

- [1. Setup](#1.-Setup)
- [2. A CT slice is a matrix of numbers](#2.-A-CT-slice-is-a-matrix-of-numbers)
- [3. Hounsfield units](#3.-Hounsfield-units)
- [4. Windowing](#4.-Windowing)
- [5. The lung mask](#5.-The-lung-mask)
- [6. Pixel spacing](#6.-Pixel-spacing)
- [7. DICOM](#7.-DICOM)
- [8. NIfTI](#8.-NIfTI)
- [9. A 3D volume](#9.-A-3D-volume)
- [10. Exercises](#10.-Exercises)
- [11. To go further](#11.-To-go-further)

## 1. Setup

We use three libraries to read medical images:

- `nibabel` reads NIfTI files (`.nii`, or `.nii` when compressed);
- `pydicom` reads DICOM files (`.dcm`);
- `numpy` and `matplotlib` to work with the numbers and to plot them.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pydicom

In [ ]:
# Folder of the course dataset (on Kaggle: right panel -> Input -> Add Input)
# On Kaggle the datasets added with "Add Input" are under /kaggle/input, in a subfolder
# (datasets/<owner>/). We look for the course dataset by its name.
DATA_DIR = next(Path("/kaggle/input").rglob("luna16-lung-slices"), Path("/kaggle/input/luna16-lung-slices"))

The dataset comes from **LUNA16**, a public collection of 888 chest CT scans. For the course we extracted
**one axial slice per patient**, so the dataset is small and fast to use:

| Folder | Content |
|---|---|
| `images/` | one CT slice per patient (`case_0001.nii`, ...) |
| `labels/` | the lung mask of each slice, same file name |
| `demo/` | one complete scan, as a DICOM series and as a NIfTI volume |
| `metadata.csv` | one row per patient (pixel spacing, slice position, ...) |

## 2. A CT slice is a matrix of numbers

We load one slice. `nib.load` opens the file, `get_fdata()` returns the pixel values as a NumPy array.

In [ ]:
image_nii = nib.load(DATA_DIR / "images" / "case_0001.nii")
image = image_nii.get_fdata()

print("dtype:", image.dtype)
print("shape:", image.shape)
print("min  :", image.min())
print("max  :", image.max())

The slice is a 512 x 512 matrix. Each number is the measurement of one small piece of the patient.

In [ ]:
# The values of a small 5 x 5 square in the middle of the image
image[250:255, 250:255]

To see the matrix as a picture we use `plt.imshow`.

In our files the first axis is x (left-right) and the second is y (front-back). Matplotlib expects rows first,
so we always plot the transposed array: **`image.T`**.

In [ ]:
plt.imshow(image.T, cmap="gray")
plt.colorbar()
plt.show()

The picture is almost uniformly grey: the range of the values is very wide and the interesting part is squeezed
into a few grey levels. To fix this we need to know what the numbers mean.

## 3. Hounsfield units

The values of a CT image are expressed in **Hounsfield units (HU)**. They measure how much each tissue attenuates
X-rays, on a scale where water is 0 and air is -1000. The scale is the same on every scanner.

| Tissue | HU (approximate) |
|---|---|
| Air | -1000 |
| Lung | -900 to -500 |
| Fat | -100 to -50 |
| Water | 0 |
| Soft tissue (muscle, organs) | 20 to 80 |
| Bone | 300 to 2000 |

The histogram of the slice shows these groups.

In [ ]:
plt.hist(image.flatten(), bins=200)
plt.xlabel("HU")
plt.ylabel("number of pixels")
plt.show()

There are two large peaks: one near -1000 (air and lung) and one near 0 (soft tissue).

**Note.** Some scans contain values far below -1000 (for example -2000 or -3024). They are not tissue: the scanner
writes them outside its circular field of view. We will always cut them away.

## 4. Windowing

A screen shows about 256 grey levels, a CT has thousands of HU values. Radiologists choose a **window**: a range of
HU that is spread over all the grey levels. Everything below the range is black, everything above is white.

With `plt.imshow` the window is set with `vmin` and `vmax`.

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.title("Lung window: -1000 to 400")
plt.imshow(image.T, cmap="gray", vmin=-1000, vmax=400)
plt.axis("off")

plt.subplot(1, 3, 2)
plt.title("Soft tissue window: -160 to 240")
plt.imshow(image.T, cmap="gray", vmin=-160, vmax=240)
plt.axis("off")

plt.subplot(1, 3, 3)
plt.title("Bone window: -500 to 1500")
plt.imshow(image.T, cmap="gray", vmin=-500, vmax=1500)
plt.axis("off")

plt.show()

The data are the same in the three pictures: only the display changes. For the rest of the course we use the
window **-1000 to 400**, where the lungs are clearly visible.

The same idea is used to prepare the image for a neural network: we cut the values to the window and rescale them
between 0 and 1.

In [ ]:
image_windowed = np.clip(image, -1000, 400)           # cut the values outside the window
image_scaled = (image_windowed + 1000) / (400 + 1000)  # rescale to 0..1

print("before:", image.min(), image.max())
print("after :", image_scaled.min(), image_scaled.max())

## 5. The lung mask

A **segmentation mask** is an image of the same size as the CT slice. Each pixel contains the label of the
structure it belongs to. In our dataset:

| Value | Meaning |
|---|---|
| 0 | background |
| 1 | right lung |
| 2 | left lung |
| 3 | trachea and main airways |

In [ ]:
mask = nib.load(DATA_DIR / "labels" / "case_0001.nii").get_fdata()

print("shape :", mask.shape)
print("values:", np.unique(mask))

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.title("Image")
plt.imshow(image.T, cmap="gray", vmin=-1000, vmax=400)
plt.axis("off")

plt.subplot(1, 3, 2)
plt.title("Mask")
plt.imshow(mask.T, cmap="viridis")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.title("Mask over the image")
plt.imshow(image.T, cmap="gray", vmin=-1000, vmax=400)
plt.imshow(np.ma.masked_equal(mask, 0).T, cmap="autumn", alpha=0.4)
plt.axis("off")

plt.show()

By convention CT slices are shown as if we looked at the patient from the feet: the **right lung is on the left
of the picture**.

In the project we want one single class, "lung". We get it by joining labels 1 and 2:

In [ ]:
lung = (mask == 1) | (mask == 2)

print("lung pixels:", lung.sum())
print("fraction of the image:", lung.mean())

**Note.** The masks of LUNA16 were not drawn by radiologists: they were produced by an automatic algorithm.
They are good, but not perfect. Keep this in mind when we evaluate our model.

## 6. Pixel spacing

A pixel has a physical size, called **pixel spacing**, written in the file header in millimetres.
It changes from scan to scan. We need it to convert pixels into real measurements.

In [ ]:
spacing_x, spacing_y = image_nii.header.get_zooms()
print("pixel spacing (mm):", spacing_x, spacing_y)

In [ ]:
pixel_area_mm2 = spacing_x * spacing_y
lung_area_cm2 = lung.sum() * pixel_area_mm2 / 100

print("lung area in this slice (cm2):", lung_area_cm2)

## 7. DICOM

**DICOM** is the format used in hospitals. A CT scan is saved as a folder with **one file per slice**.
Each file contains the pixels and a long list of information (*tags*) about the patient, the scanner and the
geometry of the image.

In [ ]:
dicom_dir = DATA_DIR / "demo" / "dicom" / "case_0001"
dicom_files = sorted(dicom_dir.glob("*.dcm"))

print("number of files:", len(dicom_files))

In [ ]:
ds = pydicom.dcmread(dicom_files[0])

print("Patient ID      :", ds.PatientID)
print("Modality        :", ds.Modality)
print("Rows, Columns   :", ds.Rows, ds.Columns)
print("Pixel spacing   :", ds.PixelSpacing)
print("Slice thickness :", ds.SliceThickness)
print("Slice position  :", ds.ImagePositionPatient)
print("Rescale slope   :", ds.RescaleSlope)
print("Rescale intercept:", ds.RescaleIntercept)

`print(ds)` shows all the tags. Real DICOM files contain the name and the date of birth of the patient:
they must be anonymised before they leave the hospital.

The pixels are in `ds.pixel_array`. They are **not** in Hounsfield units yet: the file stores raw integers,
and the header gives the formula to convert them.

`HU = pixel_array * RescaleSlope + RescaleIntercept`

In [ ]:
raw = ds.pixel_array
hu = raw * float(ds.RescaleSlope) + float(ds.RescaleIntercept)

print("raw values:", raw.min(), "to", raw.max())
print("HU values :", hu.min(), "to", hu.max())

In [ ]:
# A DICOM image is stored rows first, so here we do not transpose
plt.imshow(hu, cmap="gray", vmin=-1000, vmax=400)
plt.axis("off")
plt.show()

To build the 3D volume we read all the files, **sort them by position** along the body
(third number of `ImagePositionPatient`) and stack them.

In [ ]:
slices = [pydicom.dcmread(f) for f in dicom_files]
slices = sorted(slices, key=lambda s: float(s.ImagePositionPatient[2]))

volume_dicom = np.stack([s.pixel_array * float(s.RescaleSlope) + float(s.RescaleIntercept) for s in slices])
print("volume shape (slices, rows, columns):", volume_dicom.shape)

## 8. NIfTI

**NIfTI** (`.nii` or `.nii.gz`) is the format most used in research. The whole volume is in **one file**,
already in Hounsfield units, with a short header. It contains no patient information.
All the images of our dataset are NIfTI files.

In [ ]:
volume_nii = nib.load(DATA_DIR / "demo" / "volume" / "case_0001_image.nii")
volume = volume_nii.get_fdata()

print("shape (x, y, z)   :", volume.shape)
print("voxel size (mm)   :", volume_nii.header.get_zooms())

The voxels are not cubes: in this scan they are about 0.76 mm wide and 2.5 mm thick.

The header also contains the **affine**, a 4 x 4 matrix that converts the position of a voxel in the array
(i, j, k) into a position in the patient, in millimetres.

In [ ]:
print(volume_nii.affine)

| | DICOM | NIfTI |
|---|---|---|
| Files | one per slice | one per volume |
| Values | raw, to convert with slope and intercept | ready to use |
| Patient information | yes | no |
| Used in | hospitals, scanners | research, deep learning |

The original LUNA16 files use a third format, MetaImage (`.mhd` + `.raw`), read with the library `SimpleITK`.
We converted them to NIfTI for you.

## 9. A 3D volume

A CT scan is a stack of axial slices. Once it is a 3D array we can cut it in any direction.

In [ ]:
x, y, z = volume.shape

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.title("Axial")
plt.imshow(volume[:, :, z // 2].T, cmap="gray", vmin=-1000, vmax=400)
plt.axis("off")

plt.subplot(1, 3, 2)
plt.title("Coronal")
plt.imshow(volume[:, y // 2, :].T, cmap="gray", vmin=-1000, vmax=400, origin="lower", aspect=2.5 / 0.76)
plt.axis("off")

plt.subplot(1, 3, 3)
plt.title("Sagittal")
plt.imshow(volume[x // 2, :, :].T, cmap="gray", vmin=-1000, vmax=400, origin="lower", aspect=2.5 / 0.76)
plt.axis("off")

plt.show()

`aspect` stretches the picture to account for the slice thickness (2.5 mm) being larger than the pixel size
(0.76 mm). Without it the patient would look flattened.

## 10. Exercises

Write your code in the cell below each exercise. Everything you need is in the sections above.

### Exercise 1 - Load and show another patient

1. Load the image `case_0005.nii` from the folder `images` and save the array in `image_5`.
2. Load the mask with the same name from the folder `labels` and save the array in `mask_5`.
3. Print the shape of the image and the values contained in the mask.
4. Show the image with the lung window (-1000 to 400).

In [ ]:
# 1. Load the image (hint: nib.load(...).get_fdata(), as in section 2)
image_5 = ...

# 2. Load the mask
mask_5 = ...

# 3. Print the shape of the image and the values in the mask (hint: np.unique)

# 4. Show the image with the lung window (hint: plt.imshow with vmin and vmax, remember .T)

### Exercise 2 - Lung area

Use `image_5` and `mask_5` from exercise 1.

1. Build the boolean array `lung_5` that is `True` where the mask is 1 or 2.
2. Read the pixel spacing of this patient from the header of the image file.
3. Compute the lung area in cm² and print it.

In [ ]:
# 1. Lung = label 1 or label 2 (hint: section 5)
lung_5 = ...

# 2. Pixel spacing from the header (hint: .header.get_zooms(), section 6)
spacing_x, spacing_y = ...

# 3. Area in cm2 = number of lung pixels * area of one pixel in mm2 / 100
area_5 = ...
print(area_5)

### Exercise 3 - A windowing function

1. Complete the function `apply_window(image, hu_min, hu_max)`: it cuts the values outside the window and
   rescales the result between 0 and 1.
2. Apply it to `image_5` with the window -1000 to 400 and save the result in `image_5_scaled`.
3. Print the minimum and the maximum of `image_5_scaled`: they must be 0 and 1.

In [ ]:
# 1. Complete the function (hint: np.clip, then the formula of section 4)
def apply_window(image, hu_min, hu_max):
    image = ...
    image = ...
    return image

# 2. Apply it to image_5
image_5_scaled = ...

# 3. Print minimum and maximum

### Exercise 4 - From DICOM to Hounsfield units

1. Read the DICOM file number 20 of the list `dicom_files` with `pydicom.dcmread`.
2. Print its `InstanceNumber` and its `ImagePositionPatient`.
3. Convert its pixels to Hounsfield units and save them in `hu_20`.
4. Show `hu_20` with the soft tissue window (-160 to 240).

In [ ]:
# 1. Read the file (hint: dicom_files[20])
ds_20 = ...

# 2. Print InstanceNumber and ImagePositionPatient

# 3. Convert to HU (hint: slope and intercept, section 7)
hu_20 = ...

# 4. Show with the soft tissue window (no .T for DICOM)

### Extra - if you have finished

- **Threshold segmentation.** Lung is much darker than the rest of the body. Build a mask with `image < -400`
  and show it next to the real mask. What else did you select, besides the lungs? This is why we need something
  smarter than a threshold.
- **Coronal view with the mask.** Load `demo/volume/case_0001_label.nii` and show the lungs over the coronal
  view of section 9.

## 11. To go further

- Hounsfield units: https://radiopaedia.org/articles/hounsfield-unit
- CT windowing: https://radiopaedia.org/articles/windowing-ct
- pydicom documentation: https://pydicom.github.io/pydicom/stable/
- NiBabel, coordinate systems and affines: https://nipy.org/nibabel/coordinate_systems.html
- LUNA16 dataset: https://luna16.grand-challenge.org/Data/
- Free viewers to open DICOM and NIfTI on your computer: 3D Slicer (https://www.slicer.org/), ITK-SNAP (https://www.itksnap.org/)